# Notebook 4 — Committor Regression

Train a **LightGBM** regressor to predict the committor probability  
from full-trajectory MDTraj structural features.

**Why LightGBM?** It handles high-dimensional tabular data efficiently,  
requires no feature scaling beyond normalization, and produces interpretable  
SHAP values. Compared to linear regression, XGBoost, and RandomForest, it  
consistently achieved higher R² in our experiments.

**Downsampling:** The committor distribution is strongly bimodal (most frames  
have q≈0 or q≈1). Stratified downsampling (`src/downsample.py`) equalizes the  
class distribution across committor bins so the model learns the transition-state  
region accurately.

**Outputs:**
- `results/mdtraj_lightgbm_results/mdtraj_feature_importance.jpg`
- `results/mdtraj_lightgbm_results/mdtraj_shap_summary.jpg`
- `results/mdtraj_lightgbm_results/mdtraj_true_vs_pred_train.jpg` / `results/mdtraj_lightgbm_results/mdtraj_true_vs_pred_test.jpg`

In [ ]:
import yaml
import os
import sys
sys.path.insert(0, "..")
import pandas as pd

from src.regression import regression
from src.plot_importances import plot_importances, plot_shap, plot_true_vs_pred

RESULTS_DIR = "../results/mdtraj_lightgbm_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# --- Select system ---
with open("../config/chignolin.yaml") as f:   # change to ww_domain.yaml for WW domain
    cfg = yaml.safe_load(f)

SYSTEM = cfg["system"]
N = 40   # cluster count used when computing committors

CLUSTER_PROB_CSV = f"../data/{SYSTEM}/cluster_{N}_prob.csv"
MDTRAJ_FEATURE_CSV = "/scratch/mma9420/committor_check/chignolin_mdtraj_features_full_labeled.csv"

print(f"Committor CSV: {CLUSTER_PROB_CSV}")
print(f"MDTraj features: {MDTRAJ_FEATURE_CSV}")

In [ ]:
# Load full MDTraj feature table. frame_index is retained only for alignment checks.
features = pd.read_csv(MDTRAJ_FEATURE_CSV)
committor = pd.read_csv(CLUSTER_PROB_CSV)

if "frame_index" not in features.columns:
    raise ValueError("Expected 'frame_index' in the MDTraj feature table.")
if "Committor_prob" not in committor.columns:
    raise ValueError("Expected 'Committor_prob' in the committor CSV.")
if len(features) != len(committor):
    raise ValueError(
        f"Row mismatch: MDTraj features have {len(features)} rows, "
        f"but committor labels have {len(committor)} rows."
    )

frame_index = features["frame_index"]
X = features.drop(columns=["frame_index"])
y = committor["Committor_prob"]
valid = y.notna() & (y != "Null")

# regression() reads CLUSTER_PROB_CSV and applies the same valid-label filter
# after joining X and y, so keep X full-length here to preserve frame alignment.
df_features = X

print(f"Raw MDTraj feature table: {features.shape}")
print(f"Model feature matrix X, excluding frame_index: {X.shape}")
print(f"Committor labels: {committor.shape}")
print(f"Rows with valid committor labels: {valid.sum()}")
print(f"First/last frame_index: {frame_index.iloc[0]} / {frame_index.iloc[-1]}")

## Train the regressor

In [ ]:
# down=1 uses the full dataset; set to e.g. 0.1 for 10% stratified downsample
score, model, X_train, X_test, y_train, y_test = regression(
    df_features,
    committor_csv=CLUSTER_PROB_CSV,
    down=1
)
print(f"\nTest R²: {score:.4f}")

## Feature importance and SHAP

In [ ]:
plot_importances(model, filename=f"{RESULTS_DIR}/mdtraj_feature_importance.jpg")

In [ ]:
plot_shap(model, X_test, filename=f"{RESULTS_DIR}/mdtraj_shap_summary.jpg")

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

plot_true_vs_pred(y_train, y_pred_train,
                  filename=f"{RESULTS_DIR}/mdtraj_true_vs_pred_train.jpg",
                  title="MDTraj Train — True vs Predicted Committor")
plot_true_vs_pred(y_test, y_pred_test,
                  filename=f"{RESULTS_DIR}/mdtraj_true_vs_pred_test.jpg",
                  title="MDTraj Test — True vs Predicted Committor")

## Effect of downsampling on R²

Sweep the downsampling fraction to see how much of the data is needed.

In [ ]:
import matplotlib.pyplot as plt

fractions = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
scores = []
for frac in fractions:
    s, *_ = regression(df_features, committor_csv=CLUSTER_PROB_CSV, down=frac)
    scores.append(s)
    print(f"  down={frac:.2f}  R²={s:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(fractions, scores, 'o-')
plt.xlabel('Fraction of data used')
plt.ylabel('Test R²')
plt.title('MDTraj R² vs. downsampling fraction')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/mdtraj_r2_vs_downsampling.jpg", bbox_inches="tight", dpi=300)
plt.show()